In [ ]:
#Clean and prepare the data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
dataset_folder = Path("../dataset/MERGED_CSV")

csv_files = sorted(dataset_folder.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")
print(f"First file: {csv_files[0].name}")

df = pd.read_csv(csv_files[0])

print(f"Dataset loaded successfully")
print(f"Shape: {df.shape}")

CSV files found: 63
First file: Merged01.csv
Dataset loaded successfully
Shape: (712311, 40)


In [5]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nInfinite values:")
numeric_df = df.select_dtypes(include=[np.number])
print(np.isinf(numeric_df).sum())

Dataset shape: (712311, 40)

Missing values:
Header_Length       0
Protocol Type       0
Time_To_Live        0
Rate                0
fin_flag_number     0
syn_flag_number     0
rst_flag_number     0
psh_flag_number     0
ack_flag_number     0
ece_flag_number     0
cwr_flag_number     0
ack_count           0
syn_count           0
fin_count           0
rst_count           0
HTTP                0
HTTPS               0
DNS                 0
Telnet              0
SMTP                0
SSH                 0
IRC                 0
TCP                 0
UDP                 0
DHCP                0
ARP                 0
ICMP                0
IGMP                0
IPv                 0
LLC                 0
Tot sum             0
Min                 0
Max                 0
AVG                 0
Std                11
Tot size            0
IAT                 0
Number              0
Variance           11
Label               0
dtype: int64

Duplicate rows:
167368

Infinite values:
Header_Length       

In [9]:
# Missing values
missing = df.isnull().sum()
missing = missing[missing > 0]

# Duplicate rows
duplicates = df.duplicated().sum()

# Infinite values
numeric_df = df.select_dtypes(include=[np.number])
infinite = np.isinf(numeric_df).sum()
infinite = infinite[infinite > 0]

print("=== DATA QUALITY CHECK ===")

print("\nColumns with missing values:")
if len(missing) == 0:
    print("No missing values found")
else:
    print(missing)

print("\nDuplicate rows:")
print(duplicates)

print("\nColumns with infinite values:")
if len(infinite) == 0:
    print("No infinite values found")
else:
    print(infinite)

=== DATA QUALITY CHECK ===

Columns with missing values:
Std         11
Variance    11
dtype: int64

Duplicate rows:
167368

Columns with infinite values:
Rate    14
dtype: int64


In [11]:
# 1. Replace positive and negative infinity with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 2. Remove rows containing missing values
df.dropna(inplace=True)

# 3. Remove duplicate rows
df.drop_duplicates(inplace=True)

# Reset row index after removing rows
df.reset_index(drop=True, inplace=True)

print("Data cleaning completed.")
print("New dataset shape:", df.shape)

Data cleaning completed.
New dataset shape: (544929, 40)


In [13]:
print("=== CLEANING VERIFICATION ===")

print("\nDataset shape:")
print(df.shape)

print("\nTotal missing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

numeric_df = df.select_dtypes(include=[np.number])

print("\nTotal infinite values:")
print(np.isinf(numeric_df).sum().sum())

=== CLEANING VERIFICATION ===

Dataset shape:
(544929, 40)

Total missing values:
0

Duplicate rows:
0

Total infinite values:
0


In [14]:
# Separate input features and target label

X = df.drop(columns=["Label"])
y = df["Label"]

print("Feature matrix X shape:", X.shape)
print("Target y shape:", y.shape)

print("\nNumber of input features:", X.shape[1])
print("Target column: Label")

print("\nFirst 5 labels:")
print(y.head())

Feature matrix X shape: (544929, 39)
Target y shape: (544929,)

Number of input features: 39
Target column: Label

First 5 labels:
0    DDOS-PSHACK_FLOOD
1    MIRAI-GREIP_FLOOD
2        DOS-UDP_FLOOD
3         DNS_SPOOFING
4      DDOS-ICMP_FLOOD
Name: Label, dtype: str


In [15]:
print("=== TARGET LABEL ANALYSIS ===")

print("\nNumber of unique classes:")
print(y.nunique())

print("\nAttack classes:")
print(sorted(y.unique()))

print("\nClass distribution:")
print(y.value_counts())

print("\nClass distribution (%):")
print((y.value_counts(normalize=True) * 100).round(2))

=== TARGET LABEL ANALYSIS ===

Number of unique classes:
34

Attack classes:
['BACKDOOR_MALWARE', 'BENIGN', 'BROWSERHIJACKING', 'COMMANDINJECTION', 'DDOS-ACK_FRAGMENTATION', 'DDOS-HTTP_FLOOD', 'DDOS-ICMP_FLOOD', 'DDOS-ICMP_FRAGMENTATION', 'DDOS-PSHACK_FLOOD', 'DDOS-RSTFINFLOOD', 'DDOS-SLOWLORIS', 'DDOS-SYNONYMOUSIP_FLOOD', 'DDOS-SYN_FLOOD', 'DDOS-TCP_FLOOD', 'DDOS-UDP_FLOOD', 'DDOS-UDP_FRAGMENTATION', 'DICTIONARYBRUTEFORCE', 'DNS_SPOOFING', 'DOS-HTTP_FLOOD', 'DOS-SYN_FLOOD', 'DOS-TCP_FLOOD', 'DOS-UDP_FLOOD', 'MIRAI-GREETH_FLOOD', 'MIRAI-GREIP_FLOOD', 'MIRAI-UDPPLAIN', 'MITM-ARPSPOOFING', 'RECON-HOSTDISCOVERY', 'RECON-OSSCAN', 'RECON-PINGSWEEP', 'RECON-PORTSCAN', 'SQLINJECTION', 'UPLOADING_ATTACK', 'VULNERABILITYSCAN', 'XSS']

Class distribution:
Label
DDOS-ICMP_FLOOD            66701
DDOS-UDP_FLOOD             58084
DDOS-PSHACK_FLOOD          47890
DDOS-SYN_FLOOD             47633
DDOS-TCP_FLOOD             46978
DDOS-RSTFINFLOOD           44084
DOS-UDP_FLOOD              42954
DDOS-SY

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=== TRAIN / TEST SPLIT ===")

print("\nTraining features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining labels:", y_train.shape)
print("Testing labels:", y_test.shape)

=== TRAIN / TEST SPLIT ===

Training features: (435943, 39)
Testing features: (108986, 39)

Training labels: (435943,)
Testing labels: (108986,)


In [17]:
print("=== STRATIFICATION CHECK ===")

train_distribution = (
    y_train.value_counts(normalize=True) * 100
).round(2)

test_distribution = (
    y_test.value_counts(normalize=True) * 100
).round(2)

distribution_check = pd.DataFrame({
    "Train %": train_distribution,
    "Test %": test_distribution
})

print(distribution_check)

=== STRATIFICATION CHECK ===
                         Train %  Test %
Label                                   
DDOS-ICMP_FLOOD            12.24   12.24
DDOS-UDP_FLOOD             10.66   10.66
DDOS-PSHACK_FLOOD           8.79    8.79
DDOS-SYN_FLOOD              8.74    8.74
DDOS-TCP_FLOOD              8.62    8.62
DDOS-RSTFINFLOOD            8.09    8.09
DOS-UDP_FLOOD               7.88    7.88
DDOS-SYNONYMOUSIP_FLOOD     7.42    7.42
DOS-TCP_FLOOD               5.85    5.85
DOS-SYN_FLOOD               4.90    4.90
BENIGN                      3.04    3.04
MIRAI-GREETH_FLOOD          2.77    2.77
MIRAI-UDPPLAIN              2.44    2.44
MIRAI-GREIP_FLOOD           2.05    2.05
DDOS-ICMP_FRAGMENTATION     1.24    1.24
VULNERABILITYSCAN           1.06    1.07
MITM-ARPSPOOFING            0.81    0.81
DDOS-ACK_FRAGMENTATION      0.79    0.79
DDOS-UDP_FRAGMENTATION      0.78    0.78
DNS_SPOOFING                0.50    0.50
RECON-HOSTDISCOVERY         0.37    0.37
RECON-OSSCAN                

In [18]:
print("=== MODEL INPUT FEATURES ===\n")

for number, feature in enumerate(X.columns, start=1):
    print(f"{number:02d}. {feature}")

=== MODEL INPUT FEATURES ===

01. Header_Length
02. Protocol Type
03. Time_To_Live
04. Rate
05. fin_flag_number
06. syn_flag_number
07. rst_flag_number
08. psh_flag_number
09. ack_flag_number
10. ece_flag_number
11. cwr_flag_number
12. ack_count
13. syn_count
14. fin_count
15. rst_count
16. HTTP
17. HTTPS
18. DNS
19. Telnet
20. SMTP
21. SSH
22. IRC
23. TCP
24. UDP
25. DHCP
26. ARP
27. ICMP
28. IGMP
29. IPv
30. LLC
31. Tot sum
32. Min
33. Max
34. AVG
35. Std
36. Tot size
37. IAT
38. Number
39. Variance


In [19]:
for i, feature in enumerate(X.columns, 1):
    print(i, feature)

1 Header_Length
2 Protocol Type
3 Time_To_Live
4 Rate
5 fin_flag_number
6 syn_flag_number
7 rst_flag_number
8 psh_flag_number
9 ack_flag_number
10 ece_flag_number
11 cwr_flag_number
12 ack_count
13 syn_count
14 fin_count
15 rst_count
16 HTTP
17 HTTPS
18 DNS
19 Telnet
20 SMTP
21 SSH
22 IRC
23 TCP
24 UDP
25 DHCP
26 ARP
27 ICMP
28 IGMP
29 IPv
30 LLC
31 Tot sum
32 Min
33 Max
34 AVG
35 Std
36 Tot size
37 IAT
38 Number
39 Variance


In [20]:
print(X.columns.tolist())

['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance']


In [21]:
print("Features 26–35:\n")

for i in range(25, 35):
    print(f"{i+1}. {X.columns[i]}")

Features 26–35:

26. ARP
27. ICMP
28. IGMP
29. IPv
30. LLC
31. Tot sum
32. Min
33. Max
34. AVG
35. Std


In [22]:
print("=== FEATURE DATA TYPE CHECK ===\n")

print(X_train.dtypes)

non_numeric = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("\nNumber of non-numeric features:", len(non_numeric))

if len(non_numeric) == 0:
    print("All 39 input features are numeric.")
else:
    print("Non-numeric features:")
    print(non_numeric)

=== FEATURE DATA TYPE CHECK ===

Header_Length      float64
Protocol Type        int64
Time_To_Live       float64
Rate               float64
fin_flag_number    float64
syn_flag_number    float64
rst_flag_number    float64
psh_flag_number    float64
ack_flag_number    float64
ece_flag_number    float64
cwr_flag_number    float64
ack_count            int64
syn_count            int64
fin_count            int64
rst_count            int64
HTTP               float64
HTTPS              float64
DNS                float64
Telnet             float64
SMTP               float64
SSH                float64
IRC                float64
TCP                float64
UDP                float64
DHCP               float64
ARP                float64
ICMP               float64
IGMP               float64
IPv                float64
LLC                float64
Tot sum              int64
Min                  int64
Max                  int64
AVG                float64
Std                float64
Tot size           flo

In [23]:
print("=== FINAL PRE-TRAINING CHECK ===")

overlap = X_train.index.intersection(X_test.index)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Overlapping row indices:", len(overlap))

if len(overlap) == 0:
    print("\nTrain and test sets are completely separate. ✅")
else:
    print("\nWarning: overlap detected!")

=== FINAL PRE-TRAINING CHECK ===
Training records: 435943
Testing records: 108986
Overlapping row indices: 0

Train and test sets are completely separate. ✅


In [24]:
processed_folder = Path("../dataset/processed")
processed_folder.mkdir(exist_ok=True)

X_train.to_csv(processed_folder / "X_train.csv", index=False)
X_test.to_csv(processed_folder / "X_test.csv", index=False)
y_train.to_csv(processed_folder / "y_train.csv", index=False)
y_test.to_csv(processed_folder / "y_test.csv", index=False)

print("Processed train/test files saved successfully.")

Processed train/test files saved successfully.
